In [1]:
import tracemalloc
tracemalloc.start()

from amuse.community.ph4.interface import ph4
from amuse.community.fi.interface import Fi
from amuse.community.sse.interface import SSE

# 1. Instantiate the 4th-order Hermite N-body solver
gravity = ph4()
print(f"Gravity solver loaded: {gravity.name}")

# 2. Instantiate the SPH hydrodynamics solver
hydro = Fi()
print(f"Hydro solver loaded: {hydro.name}")

# 3. Instantiate the stellar evolution code
stellar = SSE()
print(f"Stellar Evo loaded: {stellar.name}")

# 4. Stop workers to free up memory
gravity.stop()
hydro.stop()
stellar.stop()
print("All modules successfully initialized and MPI sockets are communicating!")

AttributeError: name

In [2]:
import tracemalloc
tracemalloc.start() 

import numpy as np
from amuse.units import units, nbody_system
from amuse.datamodel import Particles
from amuse.community.ph4.interface import ph4
from amuse.community.fi.interface import Fi
from amuse.community.sse.interface import SSE
from amuse.ext.stellar_wind import new_stellar_wind
from amuse.ext.sink import new_sink_particles
from amuse.couple import bridge

def setup_galactic_center():
    # 1. Initialize Sgr A* (Supermassive Black Hole)
    sgr_a = Particles(1)
    sgr_a.mass = 4.45e6 | units.MSun
    sgr_a.position = [0, 0, 0] | units.AU
    sgr_a.velocity = [0, 0, 0] | units.km / units.s
    sgr_a.radius = 100 | units.AU 
    
    # 2. Initialize S-Stars (Inner cluster - synthetic distribution)
    s_stars = Particles(27)
    s_stars.mass = 20 | units.MSun 
    s_stars.age = 9 | units.Myr 
    s_stars.position = np.random.uniform(-5000, 5000, (27, 3)) | units.AU
    s_stars.velocity = np.random.uniform(-1000, 1000, (27, 3)) | units.km / units.s
    
    # 3. Setup Gravitational Integrator (ph4)
    converter = nbody_system.nbody_to_si(sgr_a.mass.sum(), 1000 | units.AU)
    gravity = ph4(converter)
    gravity.particles.add_particles(sgr_a)
    gravity.particles.add_particles(s_stars)
    
    # 4. Setup Stellar Evolution (SSE)
    stellar = SSE()
    stellar.particles.add_particles(s_stars)
    
    # 5. Setup Hydrodynamics (Fi)
    hydro = Fi(converter)
    hydro.parameters.use_hydro_flag = True
    hydro.parameters.isothermal_flag = False 
    
    # 6. Setup Stellar Wind
    wind = new_stellar_wind(s_stars.copy(), target_gas=hydro.gas_particles)
    
    # 7. Create Sink Particle
    sink = new_sink_particles(gravity.particles[:1])
    
    # 8. Couple Gravity and Hydrodynamics (Bridge)
    system = bridge.Bridge(use_threading=False)
    system.add_system(gravity, (hydro,))
    system.add_system(hydro, (gravity,))
    
    return system, stellar, hydro, wind, sink

def run_simulation():
    system, stellar, hydro, wind, sink = setup_galactic_center()
    
    time = 0 | units.yr
    end_time = 500 | units.yr
    dt = 1 | units.yr
    
    print("Beginning Galactic Center Integration...")
    while time < end_time:
        time += dt
        
        # A. Evolve stellar mass loss and inject SPH gas particles
        stellar.evolve_model(time)
        wind.particles.mass = stellar.particles.mass
        wind.particles.radius = stellar.particles.radius
        wind.evolve_model(time)
        
        # B. Evolve the coupled gravity-hydro system
        system.evolve_model(time)
        
        # C. Handle Bondi-Hoyle-Lyttleton accretion
        sink.accrete(hydro.gas_particles)
        
        print(f"Time: {time.value_in(units.yr):.1f} yr | BH Mass: {sink.mass.value_in(units.MSun):.6e} M_Sun")

if __name__ == '__main__':
    run_simulation()

CodeException: Exception when calling function 'initialize_code', of code 'ph4Interface', exception was 'Alias 'bool8' was removed in NumPy 2.0. Use a name without a digit at the end.'

In [2]:
import numpy as np
from amuse.units import units, constants
from amuse.community.ph4.interface import ph4
from amuse.community.fi.interface import Fi
from amuse.community.sse.interface import SSE
from amuse.couple import bridge
from amuse.datamodel import Particles, Particle

# 1. Initialize the Supermassive Black Hole (Sink Particle)
black_hole = Particle()
black_hole.mass = 4.45e6 | units.MSun
black_hole.radius = 100 | units.AU  # Predefined accretion boundary
black_hole.position = [0.0, 0.0, 0.0] | units.pc
black_hole.velocity = [0.0, 0.0, 0.0] | units.kms

# 2. Initialize S-Stars (Mock data for structural testing)
s_stars = Particles(2)
s_stars.mass = 20 | units.MSun
s_stars.radius = 4.0 | units.RSun
s_stars.position = [[0.1, 0.0, 0.0], [0.0, 0.1, 0.0]] | units.pc
s_stars.velocity = [[0.0, 200.0, 0.0], [-200.0, 0.0, 0.0]] | units.kms

# 3. Instantiate and Configure Solvers
# Gravity: 4th-order Hermite N-body
gravity = ph4()
gravity.parameters.epsilon_squared = (0.0 | units.AU)**2
gravity.particles.add_particle(black_hole)
gravity.particles.add_particles(s_stars)

# Stellar Evolution: Tracks mass-loss, radius, and temperature
stellar = SSE()
stellar.particles.add_particles(s_stars)

# Hydrodynamics: Lagrangian SPH for wind shocks
hydro = Fi()
hydro.parameters.use_hydro_flag = True
hydro.parameters.isothermal_flag = False # Must be False to allow shock-heating
gas_particles = Particles() 
# hydro.gas_particles.add_particles(gas_particles) -> Populated later by stellar winds

# 4. Bridge Coupling (Hamiltonian Operator Splitting)
system = bridge.Bridge(use_threading=False)
system.add_system(gravity, (hydro,))
system.add_system(hydro, (gravity,))
system.timestep = 1.0 | units.yr

# 5. Sink Particle Accretion Logic
def accrete_onto_black_hole(bh_particle, gas, hydro_sys):
    """
    Evaluates gas particles against the BH accretion radius.
    Removes accreted particles from the hydro solver and transfers mass/momentum.
    """
    if len(gas) == 0:
        return
    
    # Calculate radial distances from the black hole
    distances = (gas.position - bh_particle.position).lengths()
    
    # Identify gas traversing the predefined accretion boundary
    accreted_gas = gas[distances < bh_particle.radius]
    
    if len(accreted_gas) > 0:
        # Explicit transfer of mass and linear momentum to the singularity
        bh_particle.mass += accreted_gas.total_mass()
        bh_particle.momentum += accreted_gas.total_momentum()
        
        # Permanently remove the accreted gas from the SPH solver and tracking set
        hydro_sys.gas_particles.remove_particles(accreted_gas)
        gas.remove_particles(accreted_gas)

# Main Evolution Loop Outline
# time = 0.0 | units.yr
# end_time = 500.0 | units.yr
# while time < end_time:
#     stellar.evolve_model(time)
#     # -> Add stellar wind injection routine here <-
#     system.evolve_model(time)
#     accrete_onto_black_hole(black_hole, gas_particles, hydro)
#     time += system.timestep

CodeException: Exception when calling function 'initialize_code', of code 'ph4Interface', exception was 'Alias 'bool8' was removed in NumPy 2.0. Use a name without a digit at the end.'